# Tutorial 11-2: Rule Generation and Evaluation Metrics
**Course:** CSEN 140: Data Mining/Machine Learning  
**Instructor:** Dr. David C. Anastasiu

--- 
## Objective
Once frequent itemsets are identified, the next task is to extract **Association Rules**. An association rule is an implication of the form $X \to Y$, where $X$ and $Y$ are disjoint itemsets. This tutorial covers how to evaluate these rules and how to generate them efficiently from frequent itemsets.

### Key Concepts
1. **Support ($s$):** The fraction of transactions that contain both $X$ and $Y$. It represents the statistical significance of a rule.
2. **Confidence ($c$):** Measures how often items in $Y$ appear in transactions that contain $X$. It represents the predictability of the rule.
3. **Anti-monotone Property of Confidence:** For rules generated from the same frequent itemset, confidence is anti-monotonic with respect to the number of items on the Right-Hand Side (RHS). If $ABC \to D$ has low confidence, then $AB \to CD$ must also have low confidence.

In [ ]:
# Re-using our frequent itemsets (F) and their support counts (sigma) from Tutorial 11-1
all_frequent_itemsets = {
    frozenset(['Bread']): 4,
    frozenset(['Milk']): 4,
    frozenset(['Beer']): 3,
    frozenset(['Diaper']): 4,
    frozenset(['Bread', 'Milk']): 3,
    frozenset(['Bread', 'Diaper']): 3,
    frozenset(['Milk', 'Diaper']): 3,
    frozenset(['Beer', 'Diaper']): 3,
    frozenset(['Bread', 'Milk', 'Diaper']): 2, # Note: This was count 2 in lecture, not frequent at minsup=3
    frozenset(['Beer', 'Diaper', 'Milk']): 2
}

N = 5 # Total transactions
minconf = 0.6 # Minimum confidence threshold

print(f"Minimum Confidence Threshold: {minconf}")

### 1. Calculating Support and Confidence
The formulas used are:
$$s(X \to Y) = \frac{\sigma(X \cup Y)}{|T|}$$
$$c(X \to Y) = \frac{\sigma(X \cup Y)}{\sigma(X)}$$

In [ ]:
def calculate_metrics(rule_lhs, rule_rhs, itemset_counts, total_n):
    combined = rule_lhs | rule_rhs
    support_count = itemset_counts.get(combined, 0)
    lhs_count = itemset_counts.get(rule_lhs, 0)
    
    support = support_count / total_n
    confidence = support_count / lhs_count if lhs_count > 0 else 0
    
    return support, confidence

# Example: {Milk, Diaper} -> {Beer}
lhs = frozenset(['Milk', 'Diaper'])
rhs = frozenset(['Beer'])
s, c = calculate_metrics(lhs, rhs, all_frequent_itemsets, N)
print(f"Rule: {set(lhs)} -> {set(rhs)}")
print(f"Support: {s:.2f}, Confidence: {c:.2f}")

### 2. Rule Generation from a Frequent Itemset
For a frequent itemset $L$, we generate all possible binary partitions $f \to L - f$. If $|L| = k$, there are $2^k - 2$ candidate rules (excluding empty sets).

In [ ]:
import itertools

def generate_rules(frequent_itemset, itemset_counts, total_n, min_conf):
    rules = []
    k = len(frequent_itemset)
    
    # Generate all non-empty proper subsets
    for r in range(1, k):
        for lhs_tuple in itertools.combinations(frequent_itemset, r):
            lhs = frozenset(lhs_tuple)
            rhs = frequent_itemset - lhs
            
            s, c = calculate_metrics(lhs, rhs, itemset_counts, total_n)
            if c >= min_conf:
                rules.append((lhs, rhs, s, c))
    return rules

itemset_3 = frozenset(['Beer', 'Diaper', 'Milk'])
found_rules = generate_rules(itemset_3, all_frequent_itemsets, N, minconf)

print(f"Rules generated for {set(itemset_3)}:")
for lhs, rhs, s, c in found_rules:
    print(f"{set(lhs)} -> {set(rhs)} (s={s}, c={c:.2f})")

### 3. Visualizing Anti-Monotonicity of Confidence
Notice that $c(ABC \to D) \ge c(AB \to CD) \ge c(A \to BCD)$. This allows us to prune the rule lattice. If a rule with a certain RHS has low confidence, all rules that move more items to the RHS will also have low confidence.

In [ ]:
# Let's compare two rules from the same itemset
rule1_lhs = frozenset(['Milk', 'Beer'])
rule1_rhs = frozenset(['Diaper'])
_, c1 = calculate_metrics(rule1_lhs, rule1_rhs, all_frequent_itemsets, N)

rule2_lhs = frozenset(['Milk'])
rule2_rhs = frozenset(['Beer', 'Diaper'])
_, c2 = calculate_metrics(rule2_lhs, rule2_rhs, all_frequent_itemsets, N)

print(f"Rule: {set(rule1_lhs)} -> {set(rule1_rhs)} | Confidence: {c1:.2f}")
print(f"Rule: {set(rule2_lhs)} -> {set(rule2_rhs)} | Confidence: {c2:.2f}")
print(f"Is c(LHS large) >= c(LHS small)? {c1 >= c2}")

---
## Summary & Conclusion

Rule generation turns frequent itemsets into actionable insights:
1. **Metric Balance:** Support ensures the rule is statistically significant (not occurring by chance), while Confidence ensures it is a reliable predictor.
2. **Computational Strategy:** We decouple the support and confidence requirements. We first find frequent itemsets (expensive), then generate rules (cheaper).
3. **Pruning Knowledge:** Just as the Apriori principle prunes itemsets, the anti-monotone property of confidence allows us to prune rules within a single frequent itemset.

**Next Steps:** In Tutorial 11-3, we will explore **Maximal** and **Closed** itemsets to find more compact ways to represent our mining results.